# KAN-17 - Documentação de Valores Extremos

| | |
|---|---|
| **Projeto** | Análise Estatística — Spaceship Titanic (Pós-Graduação em Deep Learning — CIn/UFPE) |
| **Ticket** | [KAN-17](https://megamatic.atlassian.net/browse/KAN-17) — Documentar valores extremos sem remoção automática |
| **Responsável** | Helton Gomes Soares de Lima |
| **Revisora** | Luciana Almeida |
| **Dependência** | [KAN-15](https://megamatic.atlassian.net/browse/KAN-15) — Resumos de variáveis quantitativas |
| **Base de dados** | `data/processed/spaceship_titanic_clean.csv` — 8.693 passageiros × 22 colunas, 0 valores ausentes |
| **Entrega** | Limites LI/LS + contagem de outliers por variável (insumo da KAN-16 e da KAN-18) |

---

## 1. Contexto e ordem de execução

Esta task documenta, de forma auditável, **quais observações são consideradas valores extremos** na base tratada do *Spaceship Titanic* e **qual decisão metodológica** o projeto adota em relação a elas. O produto não é um novo dataset: é a **definição formal do critério** e o **inventário dos pontos sinalizados**, que serão reaproveitados nas etapas seguintes.

### Por que a KAN-17 deve ser executada **antes** da KAN-16 (Gráficos Descritivos)

Os boxplots da KAN-16 não são desenhos livres: o bigode (*whisker*) de um boxplot de Tukey termina exatamente em $Q_1 - 1{,}5 \times IQR$ e $Q_3 + 1{,}5 \times IQR$ — ou seja, **o boxplot já usa a regra do IQR para decidir o que é ponto e o que é bigode**, mesmo quando isso não é declarado. Se a KAN-16 fosse executada primeiro, o critério existiria de fato (escondido no *default* da biblioteca de plotagem), mas não estaria **documentado, versionado nem revisável**. Isso produziria três riscos concretos:

1. **Divergência entre figura e texto.** O relatório afirmaria "poucos outliers" enquanto o boxplot exibe centenas de pontos, ou o contrário — porque ninguém fixou o limite antes de plotar.
2. **Ambiguidade sobre o *n* analisado.** Sem decidir previamente se os extremos permanecem na base, o gráfico pode ser gerado sobre um recorte diferente do usado nas tabelas da KAN-15, e o leitor não tem como saber se o boxplot representa 8.693 ou, por exemplo, 7.755 passageiros.
3. **Risco de remoção silenciosa.** Sem uma política escrita, é comum que a etapa de gráficos "limpe" os extremos para melhorar a estética — prática que este projeto **rejeita** (ver Seção 2) e que passaria despercebida na revisão.

Fixar a regra aqui também produz o artefato que o enunciado exige: a seção de *outliers* que **interpreta** os valores atípicos como característica do fenômeno, em vez de tratá-los como sujeira a ser descartada. Além disso, a KAN-16 precisa apenas **plotar** — recebe $LI$, $LS$ e a lista de variáveis prontos, o que reduz o retrabalho e mantém a rastreabilidade exigida pela revisora.

**Ordem da trilha descritiva:** `KAN-15` (resumos quantitativos) → **`KAN-17` (esta: definição dos extremos)** → `KAN-16` (visualizações, desenhadas com estes limites) → `KAN-18` (texto consolidado).

> **Nota de consistência:** este notebook assume os mesmos parâmetros estatísticos ($Q_1$, $Q_3$, $IQR$) publicados na KAN-15. Se a KAN-15 for revisada pela Luciana, reexecute a célula de código abaixo para que tabela e gráficos permaneçam idênticos.


## 2. Fundamentação teórica

### 2.1 A regra do intervalo interquartil (IQR)

O critério adotado é o de **Tukey (1977)**, o mesmo que sustenta o boxplot. Ordenando a variável, definem-se:

- $Q_1$ — **primeiro quartil** (25º percentil): valor abaixo do qual estão 25% das observações;
- $Q_3$ — **terceiro quartil** (75º percentil): valor abaixo do qual estão 75% das observações;
- $IQR = Q_3 - Q_1$ — **amplitude interquartil**: largura do miolo central que concentra 50% dos dados;
- $LI = Q_1 - 1{,}5 \times IQR$ — **limite inferior**;
- $LS = Q_3 + 1{,}5 \times IQR$ — **limite superior**.

Uma observação é classificada como **valor extremo (outlier)** quando $x < LI$ ou $x > LS$.

O fator $1{,}5$ não é arbitrário: em uma distribuição normal, ele corresponde a aproximadamente $\pm 2{,}7$ desvios-padrão em torno da média, faixa que concentra ~99,3% dos casos. A vantagem prática do critério é ser **robusto**: depende apenas de quartis, não da média nem do desvio-padrão — que são justamente as medidas contaminadas por valores extremos. Por isso ele funciona mesmo em variáveis assimétricas, como as despesas a bordo deste dataset.

### 2.2 Por que **não** remover os outliers automaticamente

A remoção automática de observações acima de $LS$ seria inadequada neste projeto por cinco razões:

1. **Valores altos aqui são legítimos, não erros.** O dataset registra despesas reais de passageiros em serviços de bordo (Spa, VRDeck, FoodCourt, ShoppingMall, RoomService). Um gasto de US$ 30 mil não é falha de digitação: é o comportamento de um passageiro de alto poder aquisitivo — perfil de interesse comercial direto. Em dados de consumo, a cauda alta é frequentemente **o segmento mais relevante para a decisão**, e tratá-la como ruído elimina justamente a informação que o cliente quer enxergar.
2. **A distribuição é zero-inflada, o que comprime o IQR e rebaixa o $LS$.** Como a mediana das rubricas individuais é zero, $Q_1 = 0$ e o $IQR$ fica pequeno; o limite superior resultante é baixo e sinaliza uma parcela grande da base como "extrema". Aplicar o critério mecanicamente descartaria uma fração substancial da amostra — não por anomalia, mas por uma característica estrutural do fenômeno (parte da frota viaja em criossono e não consome).
3. **A remoção introduziria viés de seleção e destruiria sinal preditivo.** O consumo a bordo é uma das variáveis mais associadas ao desfecho `Transported`. Excluir os maiores gastadores alteraria a distribuição condicional do alvo, mudaria as médias e as proporções relatadas e comprometeria a validade dos testes de hipóteses e das regressões das etapas seguintes — que passariam a descrever uma população que não existe.
4. **Existem alternativas robustas que preservam a informação.** Em vez de descartar, o projeto usa mediana e IQR como medidas-resumo, representação em escala logarítmica quando a cauda longa prejudica a leitura, e — se algum teste exigir suposição de normalidade — análise de sensibilidade (resultados com e sem extremos), em vez de exclusão silenciosa.
5. **O enunciado do trabalho trata outliers como achado analítico, não como etapa de limpeza.** Ele pede explicitamente "outliers e valores faltantes" na análise descritiva, dentro de um relatório de 10–15 páginas cujo peso de clareza é 50%. Documentar os extremos, dizer quantos são, quem são e o que representam **é** a entrega; suprimi-los seria esconder um resultado.

### 2.3 Política adotada neste projeto

| Decisão | Implementação |
|---|---|
| **Manter 100% dos registros** | Nenhuma linha é removida; a base permanece com $n = 8.693$. |
| **Documentar o critério** | $LI$ e $LS$ calculados por variável, com contagem e percentual de sinalizados. |
| **Garantir consistência com a KAN-16** | Os boxplots usarão **estes** limites; os pontos exibidos são exatamente os contabilizados aqui. |
| **Preservar rastreabilidade** | O critério fica versionado no notebook e é citado no texto da KAN-18. |
| **Testar sensibilidade, se necessário** | Se um teste paramétrico exigir, as KAN-26 e KAN-27 (Daniela) comparam resultados com e sem extremos — sem alterar a base oficial. |
| **Proibido** | `df = df[df[col] <= LS]` ou qualquer filtro equivalente sobre a base de análise. |

> **Escopo das variáveis:** o critério é aplicado às **variáveis quantitativas de ampla cardinalidade** (`Age` — discreta em anos completos — e as variáveis contínuas de gasto: as cinco rubricas e `TotalSpend`). `GroupSize` e `CabinNum` ficam fora: a primeira é discreta e limitada (1 a 8) e a segunda é um identificador posicional de cabine — em ambas a regra do IQR não tem interpretação substantiva.


In [ ]:
# =============================================================================
# KAN-17 — Limites de outliers pela regra de 1,5 x IQR (Tukey)
#
# LI = Q1 - 1,5 * IQR      |      LS = Q3 + 1,5 * IQR
# Nenhuma observação é removida: a base permanece com n = 8.693.
# =============================================================================
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# --- 1. Localização resiliente da base tratada -------------------------------
# Funciona com o notebook aberto em notebooks/ (Jupyter local) ou na raiz do repo.
CANDIDATOS = [
    Path("../data/processed/spaceship_titanic_clean.csv"),
    Path("data/processed/spaceship_titanic_clean.csv"),
]
DATA_PATH = next((caminho for caminho in CANDIDATOS if caminho.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Base tratada não encontrada. Execute este notebook a partir da pasta "
        "notebooks/ ou da raiz do projeto; no Colab, faça upload de "
        "data/processed/spaceship_titanic_clean.csv."
    )

df = pd.read_csv(DATA_PATH)

# --- 2. Contratos de integridade herdados da KAN-11/KAN-12 -------------------
assert df.shape == (8693, 22), f"Shape inesperado: {df.shape}"
assert df.isna().sum().sum() == 0, "A base tratada não deve conter valores ausentes"
assert df["PassengerId"].is_unique, "PassengerId deve ser chave única"

# --- 3. Variáveis analisadas -------------------------------------------------
# Variáveis quantitativas de ampla cardinalidade (ver Seção 2.3): Age é
# discreta em anos completos; as demais são contínuas (5 rubricas + TotalSpend).
# GroupSize e CabinNum ficam fora por serem, respectivamente, discreta limitada
# e identificador posicional.
VARIAVEIS = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "TotalSpend",
]

FATOR_IQR = 1.5

# --- 4. Cálculo dos limites e contagem de outliers ---------------------------
registros = []
for var in VARIAVEIS:
    s = df[var].astype(float)

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    li = q1 - FATOR_IQR * iqr
    ls = q3 + FATOR_IQR * iqr

    abaixo_li = int((s < li).sum())
    acima_ls = int((s > ls).sum())
    n_outliers = abaixo_li + acima_ls

    registros.append({
        "Variável": var,
        "Q1 (25%)": q1,
        "Q3 (75%)": q3,
        "IQR (Q3 - Q1)": iqr,
        "LI (Q1 - 1,5·IQR)": li,
        "LS (Q3 + 1,5·IQR)": ls,
        "Abaixo do LI (n)": abaixo_li,
        "Acima do LS (n)": acima_ls,
        "Outliers (n)": n_outliers,
        "% do Dataset": n_outliers / len(df) * 100,
    })

tabela_outliers = pd.DataFrame(registros)

# --- 5. Exibição (arredondamento apenas visual; os dados brutos são mantidos) -
colunas_numericas = [
    "Q1 (25%)", "Q3 (75%)", "IQR (Q3 - Q1)",
    "LI (Q1 - 1,5·IQR)", "LS (Q3 + 1,5·IQR)", "% do Dataset",
]
exibicao = tabela_outliers.copy()
exibicao[colunas_numericas] = exibicao[colunas_numericas].round(2)

def pct(valor: float) -> str:
    """Formata percentual no padrão pt-BR (vírgula decimal)."""
    return f"{valor:.2f}%".replace(".", ",")


def moeda(valor: float) -> str:
    """Formata valor monetário no padrão pt-BR (US$ 1.234,56)."""
    return f"US$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


print(f"Base analisada: {len(df):,} passageiros × {df.shape[1]} colunas".replace(",", "."))
print(f"Critério: LI = Q1 - 1,5·IQR  |  LS = Q3 + 1,5·IQR")
print("Decisão: nenhum registro removido (ver Seção 2.2).\n")
display(exibicao)

# --- 6. Evidência complementar: quem são os extremos de TotalSpend -----------
# Ajuda a sustentar a decisão de NÃO remover (Seção 2.2). Não substitui a
# análise: use os números na seção 'Análise dos Resultados'.
ls_total = tabela_outliers.set_index("Variável").loc["TotalSpend", "LS (Q3 + 1,5·IQR)"]
extremos_total = df.loc[df["TotalSpend"] > ls_total]

print("Perfil dos extremos de TotalSpend (acima do LS):")
print(f"  n = {len(extremos_total)} ({pct(len(extremos_total) / len(df) * 100)} da base)")
print(f"  mediana do grupo = {moeda(extremos_total['TotalSpend'].median())}")
print(f"  menor valor do grupo = {moeda(extremos_total['TotalSpend'].min())}")
print(f"  VIP no grupo = {pct(extremos_total['VIP'].mean() * 100)}  |  VIP na base = {pct(df['VIP'].mean() * 100)}")
print(f"  CryoSleep no grupo = {pct(extremos_total['CryoSleep'].mean() * 100)}  |  CryoSleep na base = {pct(df['CryoSleep'].mean() * 100)}")

# --- 7. Evidência complementar citada na Seção 3 ------------------------------
# Zero-inflação das rubricas, assimetria de Age e continuidade da cauda de
# TotalSpend — números usados em 3.2 e 3.3.
print("\nEvidência complementar citada na Seção 3:")

rubricas_gasto = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
zeros = {var: (df[var] == 0).mean() * 100 for var in rubricas_gasto}
print("  Percentual de zeros nas rubricas de gasto:")
for var in rubricas_gasto:
    print(f"    {var}: {pct(zeros[var])}")
print(f"  Faixa de zeros: {pct(min(zeros.values()))} a {pct(max(zeros.values()))}")

print(f"  Assimetria (skew) de Age: {df['Age'].skew():.2f}".replace(".", ","))

distancias_total = extremos_total["TotalSpend"].sort_values().diff().dropna()
print(f"  Distância mediana entre valores consecutivos do grupo: {moeda(distancias_total.median())}")
print(f"  Maior salto entre valores consecutivos do grupo: {moeda(distancias_total.max())}")
print(f"  Máximo de TotalSpend na base: {moeda(df['TotalSpend'].max())}")

# --- 8. Nota metodológica (evita leitura incorreta dos números) --------------
print("\nNota: nas rubricas de gasto a mediana é zero e Q1 = 0, o que comprime o IQR")
print("e rebaixa o LS. Percentuais altos de sinalizados refletem a zero-inflação")
print("estrutural do consumo a bordo, não erro de dados. Por isso não há remoção.")


Base analisada: 8.693 passageiros × 22 colunas
Critério: LI = Q1 - 1,5·IQR  |  LS = Q3 + 1,5·IQR
Decisão: nenhum registro removido (ver Seção 2.2).



,Variável,Q1 (25%),Q3 (75%),IQR (Q3 - Q1),"LI (Q1 - 1,5·IQR)","LS (Q3 + 1,5·IQR)",Abaixo do LI (n),Acima do LS (n),Outliers (n),% do Dataset
0,Age,20.0,37.0,17.0,-5.5,62.5,0,162,162,1.86
1,RoomService,0.0,43.0,43.0,-64.5,107.5,0,1908,1908,21.95
2,FoodCourt,0.0,67.0,67.0,-100.5,167.5,0,1899,1899,21.85
3,ShoppingMall,0.0,23.0,23.0,-34.5,57.5,0,1891,1891,21.75
4,Spa,0.0,56.0,56.0,-84.0,140.0,0,1826,1826,21.01
5,VRDeck,0.0,41.0,41.0,-61.5,102.5,0,1860,1860,21.40
6,TotalSpend,0.0,1444.0,1444.0,-2166.0,3610.0,0,938,938,10.79


Perfil dos extremos de TotalSpend (acima do LS):
  n = 938 (10,79% da base)
  mediana do grupo = US$ 6.143,00
  menor valor do grupo = US$ 3.617,00
  VIP no grupo = 8,42%  |  VIP na base = 2,29%
  CryoSleep no grupo = 0,00%  |  CryoSleep na base = 36,06%

Evidência complementar citada na Seção 3:
  Percentual de zeros nas rubricas de gasto:
    RoomService: 65,24%
    FoodCourt: 63,83%
    ShoppingMall: 65,80%
    Spa: 62,26%
    VRDeck: 64,40%
  Faixa de zeros: 62,26% a 65,80%
  Assimetria (skew) de Age: 0,43
  Distância mediana entre valores consecutivos do grupo: US$ 6,00
  Maior salto entre valores consecutivos do grupo: US$ 4.911,00
  Máximo de TotalSpend na base: US$ 35.987,00

Nota: nas rubricas de gasto a mediana é zero e Q1 = 0, o que comprime o IQR
e rebaixa o LS. Percentuais altos de sinalizados refletem a zero-inflação
estrutural do consumo a bordo, não erro de dados. Por isso não há remoção.


## 3. Análise dos Resultados

### 3.1 Síntese dos limites encontrados

| Variável | Q1 | Q3 | LS (Q3 + 1,5·IQR) | Acima do LS (n) | % da base |
|---|---:|---:|---:|---:|---:|
| `Age` | 20 | 37 | 62,5 anos | 162 | 1,86% |
| `RoomService` | 0 | 43 | US$ 107,50 | 1.908 | 21,95% |
| `FoodCourt` | 0 | 67 | US$ 167,50 | 1.899 | 21,85% |
| `ShoppingMall` | 0 | 23 | US$ 57,50 | 1.891 | 21,75% |
| `Spa` | 0 | 56 | US$ 140,00 | 1.826 | 21,01% |
| `VRDeck` | 0 | 41 | US$ 102,50 | 1.860 | 21,40% |
| `TotalSpend` | 0 | 1.444 | US$ 3.610,00 | 938 | 10,79% |

- Variável com maior proporção de sinalizados: **`RoomService`** (21,95% da base, 1.908 passageiros acima de US$ 107,50).
- Variável com menor proporção: **`Age`** (1,86%, 162 passageiros acima de 62,5 anos).
- `TotalSpend`: $LS$ = **US$ 3.610,00** e **938** observações acima (10,79% da base).
- Observações abaixo do $LI$: **0 em todas as variáveis**, como esperado: o $LI$ é negativo em todas elas e não há valores negativos na base.

### 3.2 Por que os percentuais altos não indicam problema de dados

As cinco rubricas de gasto têm entre 62% e 66% de passageiros com valor zero. Com isso, $Q_1 = 0$ e a mediana também é zero, o que comprime o $IQR$ (de US$ 23 a US$ 67) e rebaixa o $LS$ para valores muito baixos, entre US$ 57,50 e US$ 167,50. Na prática, o critério de Tukey passa a classificar como "extremo" qualquer consumo um pouco acima de valores modestos, e é por isso que cerca de 21% da base aparece sinalizada em cada rubrica.

Esse resultado é consequência da **zero-inflação estrutural** do consumo a bordo, não de erro de digitação ou de sistema. O critério do IQR foi desenhado para distribuições aproximadamente unimodais e perde poder discriminante quando há grande massa de observações em zero. Ele serve como diagnóstico, mas não justifica a remoção de registros.

O contraste com `Age` confirma a leitura: em uma variável sem zero-inflação e aproximadamente simétrica (assimetria de 0,43), o mesmo critério sinaliza apenas 1,86% da base, um percentual compatível com o esperado.

### 3.3 Quem são os valores extremos

O grupo acima do $LS$ de `TotalSpend` (938 passageiros) tem perfil coerente com um segmento real de clientes:

- **Valores do grupo:** mediana de US$ 6.143,00 e menor valor de US$ 3.617,00, logo acima do limite. A cauda é contínua: entre valores consecutivos do grupo, a distância mediana é de apenas US$ 6, e o maior salto (US$ 4.911) ocorre só no topo, até o máximo de US$ 35.987, valor ainda plausível para um passageiro de alto consumo.
- **Status VIP:** 8,42% no grupo, contra 2,29% na base, uma concentração 3,7 vezes maior.
- **CryoSleep:** 0,00% no grupo, contra 36,06% na base. Isso é coerente com a regra de negócio, pois passageiros em criossono não consomem.

A cauda alta representa, portanto, **passageiros acordados e de maior poder aquisitivo**. Excluí-los apagaria justamente o segmento mais relevante para decisões de receita e enviesaria para baixo as estimativas de consumo e as comparações entre grupos.

### 3.4 Implicações para o relatório e para as próximas etapas

- **(a) KAN-16:** os boxplots exibirão exatamente estes pontos. Nas rubricas de gasto, recomenda-se escala logarítmica, $\log(x + 1)$, para que a caixa central não fique achatada em zero.
- **(b) Medidas-resumo:** nas variáveis de gasto, o relatório deve usar mediana e IQR, sempre acompanhados da média e do percentual de zeros. Média e desvio-padrão isolados superestimam o passageiro típico.
- **(c) Testes inferenciais:** a decisão é **manter todos os registros**. A coerência com testes robustos (Mann-Whitney, Kruskal-Wallis e Welch) será tratada nas KAN-22, 26 e 27. Se algum teste exigir normalidade, poderá ser feita uma análise de sensibilidade com e sem extremos, sem alterar a base oficial em `data/processed/`.